# Phase 3: The Champion Model — Traditional Logistic Regression Scorecard
## Project: Institutional Credit Risk Scoring Pipeline

### 1. Objective
The objective of this notebook is to train our "Champion" credit scoring model using the Weight of Evidence (WoE) transformed variables engineered in Phase 2. This linear approach serves as the industry baseline due to its total mathematical transparency and ease of regulatory auditing.

### 2. Method: Maximum Likelihood Estimation (MLE)
We fit a multivariate Logistic Regression model using `statsmodels` to optimize the log-odds of a borrower defaulting:

$$\ln\left(\frac{P(\text{Default})}{1 - P(\text{Default})}\right) = \beta_0 + \beta_1 X_{1,\text{WoE}} + \beta_2 X_{2,\text{WoE}} + \dots + \beta_k X_{k,\text{WoE}}$$

Because the input features are already linearized via the WoE transformation, the resulting coefficients ($\beta$) can be easily checked for statistical significance ($p\text{-value} < 0.05$) and economic logic (negative coefficients ensuring safe WoE profiles reduce default probability).

### 3. Scorecard Scaling Arithmetic
To make the model operational for bank branch employees and credit underwriters, we transform the raw log-odds into a point-based lookup system governed by standard banking parameters:
* **Base Score = 600 points:** The baseline score awarded when a borrower's odds of being Good vs. Bad are exactly 1:1.
* **Points to Double the Odds (PDO) = 20:** Every 20-point increase in a borrower's final score indicates their risk of default has been cut exactly in half.

### 4. Discriminatory Power Benchmarks
The final point-based scorecard is validated against the complete performance cohort using the **Gini Coefficient** and the **Kolmogorov-Smirnov (KS) Statistic**. To clear the internal risk committee hurdle for production deployment, the model must achieve a $KS \ge 30\%$, representing a solid separation between the cumulative distributions of good and bad borrowers.

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import warnings

# Suppress mixed type warnings natively
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

PROCESSED_DATA_DIR = "data/processed"
input_path = os.path.join(PROCESSED_DATA_DIR, "02_engineered_features.csv.gz")

# Force low_memory=False to clean chunk processing parsing
df_master = pd.read_csv(input_path, compression='gzip', low_memory=False)

# Isolate numeric WoE columns
woe_features = [col for col in df_master.columns if col.endswith('_WoE')]

# Ensure all values are purely numeric and fill any remaining structural gaps
df_master[woe_features] = df_master[woe_features].apply(pd.to_numeric, errors='coerce').fillna(0.0)

print(f"Dataset successfully loaded via safe parser engine.")
print(f"Cleaned dataset validated. Identified {len(woe_features)} safe WoE features for regression training.")
print(f"Features ready for training: {woe_features}")

Dataset successfully loaded via safe parser engine.
Cleaned dataset validated. Identified 21 safe WoE features for regression training.
Features ready for training: ['bin_loan_amnt_WoE', 'bin_term_WoE', 'bin_int_rate_WoE', 'bin_grade_WoE', 'bin_sub_grade_WoE', 'bin_home_ownership_WoE', 'bin_annual_inc_WoE', 'bin_verification_status_WoE', 'bin_dti_WoE', 'bin_fico_range_low_WoE', 'bin_tot_cur_bal_WoE', 'bin_total_rev_hi_lim_WoE', 'bin_acc_open_past_24mths_WoE', 'bin_avg_cur_bal_WoE', 'bin_bc_open_to_buy_WoE', 'bin_bc_util_WoE', 'bin_mo_sin_old_rev_tl_op_WoE', 'bin_mo_sin_rcnt_rev_tl_op_WoE', 'bin_mo_sin_rcnt_tl_WoE', 'bin_mort_acc_WoE', 'bin_num_actv_rev_tl_WoE']


In [2]:
# Constructing array vectors from the dynamic columns
X = df_master[woe_features]
y = df_master['target']

# 70/30 baseline performance partition
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Statsmodels requires an explicit intercept column to fit beta_0
X_train_reg = sm.add_constant(X_train)
X_test_reg = sm.add_constant(X_test)

print(f"Training Matrix Shape : {X_train_reg.shape}")
print(f"Testing Matrix Shape  : {X_test_reg.shape}")

Training Matrix Shape : (783797, 22)
Testing Matrix Shape  : (335914, 22)


In [3]:
# Fit the multivariate Logistic Regression model using Maximum Likelihood Estimation
lr_model = sm.Logit(y_train, X_train_reg).fit()
print(lr_model.summary())

# Automated Regulatory P-Value Sanity Check
p_values = lr_model.pvalues
print("\n=== REGULATORY SIGNIFICANCE SANITY CHECK ===")
for feature, p_val in p_values.items():
    status = "PASS (Significant)" if p_val < 0.05 else "CRITICAL WARNING: Insignificant (p >= 0.05)"
    print(f"{feature:<35} | p-value: {p_val:.4f} | Status: {status}")

Optimization terminated successfully.
         Current function value: 0.447835
         Iterations 6


                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               783797
Model:                          Logit   Df Residuals:                   783775
Method:                           MLE   Df Model:                           21
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.09744
Time:                        22:13:26   Log-Likelihood:            -3.5101e+05
converged:                       True   LL-Null:                   -3.8891e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const                            -1.4081      0.003   -457.980      0.000      -1.414      -1.402
bin_loan_amnt_WoE                -0.6340      0.022    -28.937      0.000 

In [4]:
import pickle
import json

ARTIFACTS_DIR = os.path.join(PROCESSED_DATA_DIR, "model_artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# ── Scorecard scaling (standard PDO arithmetic) ───────────────────────────────
PDO   = 20                                 # points to double the (good:bad) odds
factor = PDO / np.log(2)                   # ≈ 28.85
offset = 600 - (factor * np.log(50))       # 600 pts anchored at 50:1 good:bad odds

# SIGN CONVENTION — critical. statsmodels models logit(PD) = β0 + Σ βᵢ·WoEᵢ, i.e. the
# coefficients describe the log-odds of DEFAULT. A scorecard, however, must read
# "higher score = safer", with score = factor·ln((1-PD)/PD) + offset. Substituting:
#     Score = (offset - factor·β0)  +  Σ (-factor·βᵢ·WoEᵢ)
# so the intercept/base term is (offset - factor·β0) and each bin earns
# points = -factor·βᵢ·WoEᵢ. Omitting the minus sign (the previous version) made the
# safest bins score *negative* and the table no longer summed to the model's score.
intercept_coef = lr_model.params['const']
base_score = round(offset - (factor * intercept_coef))

# Load the bin-level WoE values produced in notebook 02
with open(os.path.join(PROCESSED_DATA_DIR, "woe_mappings.pkl"), "rb") as f:
    pipeline_mappings = pickle.load(f)

# Build the full banker-readable point table: one row per feature per bin
scorecard_rows = []
for col in woe_features:
    bin_col = col.replace('_WoE', '')          # "bin_int_rate_WoE" → "bin_int_rate"
    if bin_col not in pipeline_mappings:
        continue
    coef = lr_model.params[col]
    for bin_label, woe_val in pipeline_mappings[bin_col].items():
        points = round(-factor * coef * woe_val)   # higher points ⇒ safer bin
        scorecard_rows.append({
            'Feature'    : bin_col.replace('bin_', ''),
            'Bin'        : bin_label,
            'WoE'        : round(woe_val, 4),
            'Coefficient': round(coef, 4),
            'Points'     : points
        })

scorecard_df = pd.DataFrame(scorecard_rows)
print(f"Base score (intercept term): {base_score}")
print(f"A borrower's final score = base score + the points of each bin they fall into.\n")
print(scorecard_df.to_string(index=False))

# Save the point table for audit/regulatory review
scorecard_df.to_csv(os.path.join(ARTIFACTS_DIR, "scorecard_point_table.csv"), index=False)

# Save scaling constants so app.py loads them rather than re-deriving them
with open(os.path.join(ARTIFACTS_DIR, "scaling_params.json"), "w") as f:
    json.dump({'factor': factor, 'offset': offset, 'base_score': base_score, 'pdo': PDO}, f, indent=2)

Base score (intercept term): 528
A borrower's final score = base score + the points of each bin they fall into.

              Feature                   Bin     WoE  Coefficient  Points
            loan_amnt    (10000.0, 15000.0] -0.0570      -0.6340      -1
            loan_amnt    (15000.0, 21000.0] -0.1598      -0.6340      -3
            loan_amnt    (21000.0, 40000.0] -0.1838      -0.6340      -3
            loan_amnt     (499.999, 7000.0]  0.2728      -0.6340       5
            loan_amnt     (7000.0, 10000.0]  0.1481      -0.6340       3
                 term             36 months  0.2890      -0.5504       5
                 term             60 months -0.6936      -0.5504     -11
             int_rate        (11.53, 13.98]  0.0846       0.1249       0
             int_rate        (13.98, 16.99] -0.2938       0.1249       1
             int_rate        (16.99, 30.99] -0.8293       0.1249       3
             int_rate          (5.319, 8.9]  1.2307       0.1249      -4
           

In [5]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

df_test_scored = pd.DataFrame({'target': y_test})
df_test_scored['computed_PD'] = lr_model.predict(X_test_reg)

log_odds = np.log((1 - df_test_scored['computed_PD']) / df_test_scored['computed_PD'])
df_test_scored['Scorecard_Points'] = (log_odds * factor + offset).astype(int)

oos_auc = roc_auc_score(df_test_scored['target'], df_test_scored['computed_PD'])
gini = (2 * oos_auc) - 1

# KS on predicted PD directly, not on integer scorecard points
goods_pd = df_test_scored[df_test_scored['target'] == 0]['computed_PD']
bads_pd  = df_test_scored[df_test_scored['target'] == 1]['computed_PD']
ks_stat, _ = ks_2samp(goods_pd, bads_pd)

print("\n=== CHAMPION MODEL METRICS (OUT-OF-SAMPLE) ===")
print(f"ROC AUC   : {oos_auc:.4f}")
print(f"Gini      : {gini:.4f}")
print(f"KS        : {ks_stat * 100:.2f}%")
print(f"Deploy gate (KS >= 30): {'PASS' if ks_stat*100 >= 30 else 'FAIL'}")

# Save scored output
df_test_scored.to_csv(
    os.path.join(PROCESSED_DATA_DIR, "03_scored_champion_output.csv.gz"),
    index=False, compression='gzip'
)

# Save metrics so notebook 04 loads them instead of hardcoding magic numbers
champion_metrics = {
    'auc' : round(oos_auc, 4),
    'gini': round(gini, 4),
    'ks'  : round(ks_stat * 100, 2)
}
with open(os.path.join(ARTIFACTS_DIR, "champion_metrics.json"), "w") as f:
    json.dump(champion_metrics, f, indent=2)

# Save model artifact.
# remove_data() strips the ~1M-row training matrix that statsmodels otherwise pickles,
# shrinking champion_scorecard.pkl from ~420 MB to a few KB. Coefficients (.params, with
# their feature-name index) and .predict() are preserved — all app.py / notebook 06 need.
lr_model.remove_data()
with open(os.path.join(ARTIFACTS_DIR, "champion_scorecard.pkl"), "wb") as f:
    pickle.dump(lr_model, f)

print(f"\nArtifacts saved: champion_scorecard.pkl (slim), scorecard_point_table.csv, champion_metrics.json")


=== CHAMPION MODEL METRICS (OUT-OF-SAMPLE) ===
ROC AUC   : 0.7145
Gini      : 0.4291
KS        : 31.07%
Deploy gate (KS >= 30): PASS



Artifacts saved: champion_scorecard.pkl (slim), scorecard_point_table.csv, champion_metrics.json
